# Essential SQL on the Airbnb clone database

This notebook walks through the SQL you reach for most often: **filtering, aggregating, joining, CTEs, window functions, arrays, date ranges and query plans**. Every query runs against the real PostgreSQL database behind the .NET backend, and each one ends with a short analysis of what the result says.

**What's in the database.** The backend is a modular monolith: each module owns its own **schema** inside the one `airbnb` database.

| Schema | Owns | Main table |
|---|---|---|
| `stays` | homes and cities | `stays.listings`, `stays.cities` |
| `experiences` | activities | `experiences.experiences` |
| `services` | bookable services | `services.services` |
| `hosts` | host profiles | `hosts.hosts` |
| `reviews` | guest reviews | `reviews.reviews` |
| `identity` | accounts and sessions | `identity.users`, `identity.sessions` |
| `bookings` | reservations | `bookings.bookings` |

> **Safe to run.** The connection is opened **read-only**, so nothing in this notebook can change data.

## 0. Setup

1. Start the stack (`aspire run` from `backend/`) so Postgres is up.
2. Install the packages: `pip install -r notebooks/requirements.txt`.
3. Put the connection URL in the `AIRBNB_DB_URL` environment variable. **Never paste the password into the notebook.**

```text
postgresql://postgres:<url-encoded password>@127.0.0.1:<port>/airbnb
```

Use `127.0.0.1` rather than `localhost`: Docker publishes the port on IPv4 only, and `localhost` can resolve to IPv6 first and hang. Find the port with `docker ps --filter name=postgres`.

In [1]:
import os

import pandas as pd
import psycopg
from psycopg.types.numeric import FloatLoader

DB_URL = os.environ.get("AIRBNB_DB_URL")
if not DB_URL:
    raise RuntimeError("Set AIRBNB_DB_URL to the database URL first (see the setup notes above).")

conn = psycopg.connect(DB_URL, autocommit=True, connect_timeout=5)
conn.read_only = True                                     # nothing here can write
conn.adapters.register_loader("numeric", FloatLoader)     # numeric -> float, so pandas shows plain numbers

pd.set_option("display.max_colwidth", 60)


def sql(query: str, params=None) -> pd.DataFrame:
    # Run one query and return its rows as a DataFrame.
    with conn.cursor() as cur:
        cur.execute(query, params)
        return pd.DataFrame(cur.fetchall(), columns=[col.name for col in cur.description])


sql("select current_database() as database, version() as server")

,database,server
0,airbnb,PostgreSQL 18.3 (Debian 18.3-1.pgdg13+1) on x86_64-pc-li...


### A naming rule you'll hit immediately

EF Core created the columns in **PascalCase** (`PricePerNight`, `HostId`). PostgreSQL folds unquoted names to lower case, so these columns **must be double-quoted**:

```sql
select "PricePerNight" from stays.listings   -- works
select PricePerNight   from stays.listings   -- error: column "pricepernight" does not exist
```

Owned types are flattened into prefixed columns, for example `"Location_City"`.

## 1. Getting oriented: how much data is there?

`count(*)` per table, stitched together with `union all`. This is the first query to run against any unfamiliar database.

In [2]:
sql('''
    select 'stays.listings'          as "table", count(*) as "rows" from stays.listings
    union all select 'stays.cities',            count(*) from stays.cities
    union all select 'hosts.hosts',             count(*) from hosts.hosts
    union all select 'experiences.experiences', count(*) from experiences.experiences
    union all select 'services.services',       count(*) from services.services
    union all select 'reviews.reviews',         count(*) from reviews.reviews
    union all select 'identity.users',          count(*) from identity.users
    union all select 'bookings.bookings',       count(*) from bookings.bookings
''')

,table,rows
0,stays.listings,16
1,stays.cities,6
2,hosts.hosts,16
3,experiences.experiences,12
4,services.services,12
5,reviews.reviews,38
6,identity.users,2
7,bookings.bookings,0


**Analysis.** The catalog is small and fixed: 16 stays, 12 experiences and 12 services, all seeded from the frontend's mock data. `reviews`, `identity` and `bookings` hold what people actually do in the app, so their counts grow with use. While `bookings` is empty, the booking queries below return no rows. Section 8 shows the same date logic on inline sample data so you can still see it work.

## 2. `SELECT … WHERE … ORDER BY … LIMIT`: the everyday query

The five cheapest stays that fit a party of 4. The guest count is passed as a **parameter** (`%(guests)s`) instead of being formatted into the string. Parameters are what keep SQL injection out of real code.

In [3]:
sql('''
    select "Id", "Title", "Location_City" as city, "PricePerNight" as price, "MaxGuests" as max_guests, "Rating"
    from stays.listings
    where "MaxGuests" >= %(guests)s
    order by "PricePerNight", "Rating" desc
    limit 5
''', {"guests": 4})

,Id,Title,city,price,max_guests,Rating
0,l7,Trending loft in the city,Lisbon,145.0,4,4.74
1,l10,Rolling-hills farmhouse,Athens,160.0,6,4.77
2,l8,A-frame cabin escape,Aspen,175.0,4,4.85
3,l3,Stone cottage in the countryside,Wilmington,180.0,4,4.79
4,l14,Designer city apartment,Lisbon,199.0,4,4.82


**Analysis.** For four guests, Lisbon's city loft is the cheapest option. Price alone flatters small places, though: to compare value fairly you need **price per guest** (section 5).

## 3. `GROUP BY` and `HAVING`: summarising by city

Aggregates collapse rows into one row per group. `WHERE` filters rows **before** grouping; `HAVING` filters groups **after**.

In [4]:
sql('''
    select "Location_City"                 as city,
           count(*)                        as listings,
           round(avg("PricePerNight"), 0)  as avg_price,
           min("PricePerNight")            as min_price,
           max("PricePerNight")            as max_price,
           round(avg("Rating"), 2)         as avg_rating
    from stays.listings
    group by "Location_City"
    having count(*) >= 2
    order by avg_price desc
''')

,city,listings,avg_price,min_price,max_price,avg_rating
0,Malibu,3,442.0,365.0,540.0,4.86
1,Athens,2,235.0,160.0,310.0,4.86
2,Lisbon,3,225.0,145.0,330.0,4.82
3,Aspen,4,224.0,175.0,260.0,4.89
4,Kyoto,2,190.0,95.0,285.0,4.89
5,Wilmington,2,134.0,88.0,180.0,4.75


**Analysis.**
- **Malibu is the premium market.** Its average nightly price (about $442) is roughly three times Wilmington's (about $134).
- **Ratings barely move with price.** Every city averages between 4.75 and 4.89, so in this catalog paying more buys location, not higher satisfaction.
- **Kyoto has the widest spread** ($95–$285): one budget option and one premium option, with nothing in between.

## 4. `JOIN` across schemas: are superhosts different?

Modules can't read each other's tables in the application code, but SQL can join across schemas freely, which is exactly what analysis needs. `stays.listings."HostId"` points at `hosts.hosts."Id"`.

In [5]:
sql('''
    select h."IsSuperhost"                    as superhost,
           count(*)                           as listings,
           round(avg(l."Rating"), 3)          as avg_rating,
           round(avg(l."PricePerNight"), 0)   as avg_price,
           round(avg(h."ResponseRate"), 0)    as avg_response_rate
    from stays.listings l
    join hosts.hosts h on h."Id" = l."HostId"
    group by h."IsSuperhost"
    order by superhost desc
''')

,superhost,listings,avg_rating,avg_price,avg_response_rate
0,True,10,4.896,315.0,98.0
1,False,6,4.772,145.0,89.0


**Analysis.** Superhosts run 10 of the 16 listings. Their listings rate higher (about **4.90 vs 4.77**), cost about **twice as much** (about $315 vs $145 a night), and the hosts answer more messages (**98% vs 89%** response rate). With only 16 rows this is a **correlation, not proof**: the premium Malibu villas may simply belong to superhosts. Section 6 controls for city.

## 5. `CASE`: turning numbers into categories

`CASE` buckets a continuous value into labels. Combined with `GROUP BY`, it gives a distribution. Here it also shows **price per guest**, a fairer measure of value.

In [6]:
sql('''
    select case
             when "PricePerNight" < 150 then '1. budget (< $150)'
             when "PricePerNight" < 300 then '2. mid ($150-299)'
             else                            '3. premium ($300+)'
           end                                                  as price_band,
           count(*)                                             as listings,
           round(avg("PricePerNight" / "MaxGuests"), 1)         as avg_price_per_guest,
           round(avg("Rating"), 2)                              as avg_rating
    from stays.listings
    group by price_band
    order by price_band
''')

,price_band,listings,avg_price_per_guest,avg_rating
0,1. budget (< $150),3,42.6,4.75
1,2. mid ($150-299),8,49.3,4.86
2,3. premium ($300+),5,61.0,4.89


**Analysis.** Half the catalog (8 of 16) sits in the mid band. A premium night costs about **3.6×** a budget one (roughly $393 vs $109), but **per guest only about 1.4×** ($61 vs $43), because premium places sleep more people. For group trips, value per guest is the better ranking. Ratings rise slightly with price (4.75 → 4.89).

## 6. CTEs and window functions: ranking within a group

A **CTE** (`with … as`) names an intermediate result so the query reads top-down. **Window functions** (`over (partition by …)`) compute across related rows *without* collapsing them, unlike `GROUP BY`. Here each listing is compared with its own city's average.

In [7]:
sql('''
    with priced as (
        select "Id", "Title", "Location_City" as city, "PricePerNight" as price,
               avg("PricePerNight") over (partition by "Location_City")             as city_avg,
               rank()               over (partition by "Location_City"
                                          order by "PricePerNight" desc)           as price_rank_in_city
        from stays.listings
    )
    select city, price_rank_in_city as "rank", "Title", price,
           round(city_avg, 0)                              as city_avg,
           round(100.0 * (price - city_avg) / city_avg, 0) as pct_vs_city_avg
    from priced
    order by city, price_rank_in_city
''')

,city,rank,Title,price,city_avg,pct_vs_city_avg
0,Aspen,1,Lakefront retreat with dock,260.0,224.0,16.0
1,Aspen,2,Forest cabin with hot tub,240.0,224.0,7.0
2,Aspen,3,Cozy cabin in the pines,220.0,224.0,-2.0
3,Aspen,4,A-frame cabin escape,175.0,224.0,-22.0
4,Athens,1,Cliffside home with amazing views,310.0,235.0,32.0
5,Athens,2,Rolling-hills farmhouse,160.0,235.0,-32.0
6,Kyoto,1,Panoramic mountain lookout,285.0,190.0,50.0
7,Kyoto,2,Tiny home in the woods,95.0,190.0,-50.0
8,Lisbon,1,Lakeside glass house,330.0,225.0,47.0
9,Lisbon,2,Designer city apartment,199.0,225.0,-11.0


**Analysis.** Within a city, prices still vary widely. Kyoto's top listing is about 50% above its city average while its budget option is about 50% below it. This per-city view is what the host earnings estimate needs: comparing a new Kyoto listing with the *overall* average would mislead.

## 7. Arrays: `unnest` and data-quality checks

`"Amenities"` is a PostgreSQL `text[]`. `unnest` expands an array into one row per element, so ordinary aggregates work on it. The second query checks a **denormalised counter** (`"ReviewCount"`) against the rows that actually exist.

In [8]:
sql('''
    select amenity, count(*) as listings
    from stays.listings, unnest("Amenities") as amenity
    group by amenity
    order by listings desc, amenity
''')

,amenity,listings
0,Air conditioning,16
1,Free parking,16
2,Kitchen,16
3,Self check-in,16
4,Washer,16
5,Wifi,16


In [9]:
sql('''
    select l."Id", l."Title", l."ReviewCount" as review_count_column,
           count(r."Id")                    as review_rows_stored,
           round(avg(r."Rating"), 2)        as avg_stored_rating
    from stays.listings l
    left join reviews.reviews r on r."SubjectType" = 'stay' and r."SubjectId" = l."Id"
    group by l."Id", l."Title", l."ReviewCount"
    order by l."ReviewCount" desc
    limit 8
''')

,Id,Title,review_count_column,review_rows_stored,avg_stored_rating
0,l4,Cliffside home with amazing views,201,2,5.0
1,l11,Panoramic mountain lookout,188,2,5.0
2,l2,Beachfront villa with infinity pool,142,2,5.0
3,l13,Lakeside glass house,134,2,5.0
4,l15,Forest cabin with hot tub,119,2,5.0
5,l6,Lakefront retreat with dock,110,2,5.0
6,l16,Sunset beach cottage,103,2,5.0
7,l9,Ocean-view beach bungalow,96,2,5.0


**Analysis.**
- **Amenities have no variance.** All 16 listings share the same six amenities, so amenity filters can't distinguish anything yet. This comes from the seed data; listings created by hosts can add more.
- **`ReviewCount` is not a count of stored rows.** The cliffside home shows 201 reviews while 2 rows exist, and the cozy cabin (`l1`) shows 88 with 4. The seeded counts are display numbers, and only *new* reviews increment them through the review messaging flow. Treat the column as the source of truth for display and the `reviews` table as a sample. The `left join` matters here: an inner join would silently drop listings that have no review rows.
- **Ratings are uniformly high.** Stored reviews average 5.0, so they carry no signal for ranking yet.

## 8. Dates and ranges: bookings and overlaps

Bookings are half-open date ranges `[CheckIn, CheckOut)`: the check-out day isn't a night stayed. PostgreSQL has a native **`daterange`** type with an overlap operator **`&&`**. The database uses exactly this in an exclusion constraint, so two confirmed stays on one listing can never share a night.

First, the live booking summary. It's empty until someone books in api mode.

In [10]:
sql('''
    select date_trunc('month', "CheckIn")::date as month,
           count(*)                            as bookings,
           sum("Nights")                       as nights,
           sum("Total")                        as revenue,
           round(avg("Nights"), 1)             as avg_stay_nights
    from bookings.bookings
    where "Status" = 'confirmed'
    group by month
    order by month
''')

,month,bookings,nights,revenue,avg_stay_nights


The same date logic, run on **inline sample rows** (`values (…)`), so you can see it work without writing anything. The query pairs every two stays of one listing and reports whether their nights overlap.

In [11]:
sql('''
    with sample(id, listing, check_in, check_out) as (
        values ('A', 'l1', date '2031-02-01', date '2031-02-05'),
               ('B', 'l1', date '2031-02-05', date '2031-02-08'),  -- starts on A's check-out day
               ('C', 'l1', date '2031-02-03', date '2031-02-06'),  -- inside A
               ('D', 'l2', date '2031-02-03', date '2031-02-06')   -- another listing
    )
    select a.id as stay, b.id as other_stay, a.listing,
           daterange(a.check_in, a.check_out, '[)') && daterange(b.check_in, b.check_out, '[)') as overlaps
    from sample a
    join sample b on a.listing = b.listing and a.id < b.id
    order by stay, other_stay
''')

,stay,other_stay,listing,overlaps
0,A,B,l1,False
1,A,C,l1,True
2,B,C,l1,True


**Analysis.** A–B **don't** overlap: B checks in the day A checks out, which is a valid back-to-back turnover. A–C and B–C **do**, so the database would refuse C. D is on another listing and is never compared. This is the same rule as the `bookings_no_overlap` exclusion constraint, which also enforces it under concurrent inserts.

`generate_series` expands each stay into the individual nights it occupies, which is how an occupancy calendar is built:

In [12]:
sql('''
    with sample(id, check_in, check_out) as (
        values ('A', date '2031-02-01', date '2031-02-05'),
               ('B', date '2031-02-05', date '2031-02-08')
    )
    select id, night::date as night
    from sample, generate_series(check_in, check_out - 1, interval '1 day') as night
    order by night
''')

,id,night
0,A,2031-02-01
1,A,2031-02-02
2,A,2031-02-03
3,A,2031-02-04
4,B,2031-02-05
5,B,2031-02-06
6,B,2031-02-07


## 9. Reading query plans with `EXPLAIN`

`EXPLAIN` shows how PostgreSQL will run a query: which indexes it uses and what the joins cost. Below is the plan for the availability lookup the calendar runs on every listing page.

In [13]:
plan = sql('''
    explain
    select "CheckIn", "CheckOut"
    from bookings.bookings
    where "ListingId" = 'l1' and "Status" = 'confirmed' and "CheckOut" > current_date
    order by "CheckIn"
''')
print("\n".join(plan.iloc[:, 0]))

Index Scan using "IX_bookings_ListingId_CheckIn" on bookings  (cost=0.14..8.17 rows=1 width=8)
  Index Cond: (("ListingId")::text = 'l1'::text)
  Filter: ((("Status")::text = 'confirmed'::text) AND ("CheckOut" > CURRENT_DATE))


**Analysis.** The planner uses an **index scan** on `IX_bookings_ListingId_CheckIn`. The `Index Cond` jumps straight to one listing's rows, which already come back sorted by `CheckIn`, so the `order by` costs nothing extra. `Status` and `CheckOut` are applied as a `Filter` on those few rows, which is cheap because one listing has few bookings. On a tiny table the planner may switch to a sequential scan, which is fine. Use `explain analyze` (it really executes the query) to compare estimated and actual row counts when tuning.

Indexes on the bookings table. `bookings_no_overlap` is the **GiST index behind the exclusion constraint** from section 8; the B-tree indexes serve the guest, host and listing lookups.

In [14]:
sql('''
    select indexname, indexdef
    from pg_indexes
    where schemaname = 'bookings' and tablename = 'bookings'
    order by indexname
''')

,indexname,indexdef
0,IX_bookings_GuestId,"CREATE INDEX ""IX_bookings_GuestId"" ON bookings.bookings ..."
1,IX_bookings_HostId,"CREATE INDEX ""IX_bookings_HostId"" ON bookings.bookings U..."
2,IX_bookings_ListingId_CheckIn,"CREATE INDEX ""IX_bookings_ListingId_CheckIn"" ON bookings..."
3,PK_bookings,"CREATE UNIQUE INDEX ""PK_bookings"" ON bookings.bookings U..."
4,bookings_no_overlap,CREATE INDEX bookings_no_overlap ON bookings.bookings US...


## 10. Cheat sheet and takeaways

| Need | Reach for |
|---|---|
| Filter rows | `where`, with **parameters**, never string formatting |
| One row per group | `group by` + aggregates, `having` to filter groups |
| Combine tables (even across schemas) | `join`; `left join` to keep rows with no match |
| Categories from numbers | `case when … then … end` |
| Compare a row with its group | window functions: `avg(…) over (partition by …)`, `rank()` |
| Readable multi-step logic | CTEs: `with step as (…) select …` |
| Array columns | `unnest(...)` |
| Date ranges and overlaps | `daterange(start, end, '[)')` with `&&`, and `generate_series` |
| Why is it slow? | `explain` / `explain analyze`, and `pg_indexes` |

**What the data says.**
- **City drives price far more than quality.** Malibu costs about three times Wilmington, while ratings stay between 4.75 and 4.89 everywhere.
- **Superhost listings rate higher and cost about twice as much.** The sample is small; control for city before drawing conclusions.
- **Watch the denormalised counters.** `ReviewCount` is a display figure; the `reviews` table is a sample. Seeded amenities and ratings carry no signal yet.
- **Bookings are empty until people book.** The overlap rule is enforced by the database itself, so the analysis can trust there are no double bookings.

In [15]:
conn.close()